# Phase 4 - Hyperparameter Tuning: Model Optimization & Validation

This notebook explores alternative optimization strategies to improve precision while maintaining 100% recall.

**Context**: Extensive LightGBM hyperparameter tuning (Notebook 02) degraded performance from F1=0.632 to F1=0.600.

**Objective**: Improve precision from 46% to 70%+ while maintaining 95%+ recall using:
1. Prediction threshold optimization
2. Feature importance analysis and selection
3. Lone Wolf validation (real-world test)
 
**Decision Point**: If these quick wins don't achieve target performance, proceed to extensive tuning of all 4 algorithms.


In [13]:
# Cell 1: Import Libraries and Configuration
import pandas as pd
import numpy as np
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, roc_curve, auc,
    f1_score, precision_score, recall_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Paths
TRAINING_DATA_PATH = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Features/all_cases_combined_features.csv'
LONEWOLF_LOGFILE = '/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-LogFile.csv'
LONEWOLF_USNJRNL = '/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-UsnJrnl.csv'
LONEWOLF_SUSPICIOUS = '/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-Suspicious.csv'
OUTPUT_DIR = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/'

# Configuration
RANDOM_STATE = 42
TARGET_RECALL = 0.95  # Minimum acceptable recall
TARGET_PRECISION = 0.70  # Target precision

print("="*80)
print("MODEL OPTIMIZATION & VALIDATION")
print("="*80)
print(f"\nTarget Performance:")
print(f"  Recall:    ≥ {TARGET_RECALL:.0%}")
print(f"  Precision: ≥ {TARGET_PRECISION:.0%}")
print(f"  F1-Score:  ≥ {2 * TARGET_PRECISION * TARGET_RECALL / (TARGET_PRECISION + TARGET_RECALL):.3f}")


MODEL OPTIMIZATION & VALIDATION

Target Performance:
  Recall:    ≥ 95%
  Precision: ≥ 70%
  F1-Score:  ≥ 0.806


In [14]:
# Cell 2: Load Training Data
print("="*80)
print("LOADING TRAINING DATA")
print("="*80)

# Load training data
df = pd.read_csv(TRAINING_DATA_PATH)

print(f"Dataset loaded successfully")
print(f"Total samples: {len(df):,}")

# Remove label leakage features
leakage_features = [
    'is_flagged_suspicious',
    'has_logfile_suspicious', 
    'has_usnjrnl_suspicious',
    'cross_artifact_detected'
]

print(f"\nRemoving {len(leakage_features)} label leakage features...")
df = df.drop(leakage_features, axis=1, errors='ignore')

# Convert boolean to int
bool_cols = df.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    df[bool_cols] = df[bool_cols].astype(int)

# Select numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'ground_truth_label' in numeric_cols:
    numeric_cols.remove('ground_truth_label')

# Create feature matrix and labels
X = df[numeric_cols]
y = df['ground_truth_label']

feature_names = numeric_cols

print(f"\nFeature columns: {len(feature_names)}")
print(f"Suspicious files: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.2f}%)")
print(f"Benign files: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")


LOADING TRAINING DATA
Dataset loaded successfully
Total samples: 88,190

Removing 4 label leakage features...

Feature columns: 30
Suspicious files: 266 (0.30%)
Benign files: 87,924 (99.70%)


In [15]:
# Cell 3: Train Base Models (All 4 Algorithms)
print("="*80)
print("TRAINING BASE MODELS (ALL 4 ALGORITHMS)")
print("="*80)

# Calculate class weight
class_weight_ratio = (y==0).sum() / (y==1).sum()
print(f"\nClass imbalance ratio: {class_weight_ratio:.2f}:1")

# Train all 4 algorithms with base parameters
print("\nTraining models...")

# 1. Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X, y)
rf_proba = rf_model.predict_proba(X)[:, 1]
print("✓ Random Forest trained")

# 2. XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=class_weight_ratio,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X, y)
xgb_proba = xgb_model.predict_proba(X)[:, 1]
print("✓ XGBoost trained")

# 3. LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=100,
    scale_pos_weight=class_weight_ratio,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgbm_model.fit(X, y)
lgbm_proba = lgbm_model.predict_proba(X)[:, 1]
print("✓ LightGBM trained")

# 4. Logistic Regression
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
lr_model.fit(X, y)
lr_proba = lr_model.predict_proba(X)[:, 1]
print("✓ Logistic Regression trained")

print("\nAll models trained successfully!")


TRAINING BASE MODELS (ALL 4 ALGORITHMS)

Class imbalance ratio: 330.54:1

Training models...
✓ Random Forest trained
✓ XGBoost trained
✓ LightGBM trained
✓ Logistic Regression trained

All models trained successfully!


In [16]:
# Cell 4: Baseline Performance (Default 0.5 Threshold)
print("="*80)
print("BASELINE PERFORMANCE (DEFAULT 0.5 THRESHOLD)")
print("="*80)

models = {
    'Random Forest': (rf_model, rf_proba),
    'XGBoost': (xgb_model, xgb_proba),
    'LightGBM': (lgbm_model, lgbm_proba),
    'Logistic Regression': (lr_model, lr_proba)
}

baseline_results = []

for name, (model, proba) in models.items():
    y_pred = (proba >= 0.5).astype(int)
    
    recall = recall_score(y, y_pred)
    precision = precision_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    
    baseline_results.append({
        'Algorithm': name,
        'Recall': recall,
        'Precision': precision,
        'F1-Score': f1,
        'Threshold': 0.5
    })
    
baseline_df = pd.DataFrame(baseline_results).sort_values('F1-Score', ascending=False)

print("\n")
print(baseline_df.to_string(index=False))

# Identify best baseline model
best_baseline_idx = baseline_df['F1-Score'].idxmax()
best_baseline_model = baseline_df.loc[best_baseline_idx, 'Algorithm']
best_baseline_f1 = baseline_df.loc[best_baseline_idx, 'F1-Score']

print(f"\nBest Baseline: {best_baseline_model} (F1: {best_baseline_f1:.4f})")



BASELINE PERFORMANCE (DEFAULT 0.5 THRESHOLD)


          Algorithm  Recall  Precision  F1-Score  Threshold
      Random Forest 1.00000   0.446309  0.617169        0.5
            XGBoost 1.00000   0.446309  0.617169        0.5
Logistic Regression 1.00000   0.406107  0.577633        0.5
           LightGBM 0.87218   0.016695  0.032764        0.5

Best Baseline: Random Forest (F1: 0.6172)


In [17]:
# Cell 5: Threshold Optimization - Find Optimal Cutoff

print("="*80)
print("THRESHOLD OPTIMIZATION")
print("="*80)

print("\nFinding optimal prediction threshold for each algorithm...")
print("Goal: Maximize F1 while maintaining recall ≥ 95%")

threshold_results = []

for name, (model, proba) in models.items():
    print(f"\n{name}:")
    
    # Calculate precision-recall curve
    precisions, recalls, thresholds = precision_recall_curve(y, proba)
    
    # For each threshold, calculate F1 score
    f1_scores = []
    valid_thresholds = []
    
    for i, threshold in enumerate(thresholds):
        if recalls[i] >= TARGET_RECALL:  # Only consider thresholds with recall ≥ 95%
            f1 = 2 * (precisions[i] * recalls[i]) / (precisions[i] + recalls[i]) if (precisions[i] + recalls[i]) > 0 else 0
            f1_scores.append(f1)
            valid_thresholds.append(threshold)
    
    if len(f1_scores) > 0:
        # Find best threshold
        best_idx = np.argmax(f1_scores)
        best_threshold = valid_thresholds[best_idx]
        best_f1 = f1_scores[best_idx]
        
        # Get metrics at best threshold
        y_pred_optimal = (proba >= best_threshold).astype(int)
        optimal_recall = recall_score(y, y_pred_optimal)
        optimal_precision = precision_score(y, y_pred_optimal)
        optimal_f1 = f1_score(y, y_pred_optimal)
        
        threshold_results.append({
            'Algorithm': name,
            'Optimal Threshold': best_threshold,
            'Recall': optimal_recall,
            'Precision': optimal_precision,
            'F1-Score': optimal_f1
        })
        
        print(f"  Optimal Threshold: {best_threshold:.4f}")
        print(f"  Recall:    {optimal_recall:.4f} ({(y_pred_optimal[y==1] == 1).sum()}/{(y==1).sum()})")
        print(f"  Precision: {optimal_precision:.4f}")
        print(f"  F1-Score:  {optimal_f1:.4f}")
    else:
        print(f"  No threshold found with recall ≥ {TARGET_RECALL:.0%}")

threshold_df = pd.DataFrame(threshold_results).sort_values('F1-Score', ascending=False)

print("\n" + "="*80)
print("THRESHOLD OPTIMIZATION RESULTS")
print("="*80)
print("\n")
print(threshold_df.to_string(index=False))

# Compare with baseline
print("\n" + "="*80)
print("IMPROVEMENT OVER BASELINE")
print("="*80)

for name in threshold_df['Algorithm']:
    baseline_f1 = baseline_df[baseline_df['Algorithm'] == name]['F1-Score'].values[0]
    threshold_f1 = threshold_df[threshold_df['Algorithm'] == name]['F1-Score'].values[0]
    improvement = threshold_f1 - baseline_f1
    
    print(f"\n{name}:")
    print(f"  Baseline F1:  {baseline_f1:.4f}")
    print(f"  Optimized F1: {threshold_f1:.4f}")
    print(f"  Improvement:  {improvement:+.4f} ({improvement/baseline_f1*100:+.2f}%)")


THRESHOLD OPTIMIZATION

Finding optimal prediction threshold for each algorithm...
Goal: Maximize F1 while maintaining recall ≥ 95%

Random Forest:
  Optimal Threshold: 0.7184
  Recall:    1.0000 (266/266)
  Precision: 0.4463
  F1-Score:  0.6172

XGBoost:
  Optimal Threshold: 0.9947
  Recall:    1.0000 (266/266)
  Precision: 0.4463
  F1-Score:  0.6172

LightGBM:
  Optimal Threshold: 0.0000
  Recall:    1.0000 (266/266)
  Precision: 0.0030
  F1-Score:  0.0060

Logistic Regression:
  Optimal Threshold: 0.9741
  Recall:    0.9850 (262/266)
  Precision: 0.4260
  F1-Score:  0.5948

THRESHOLD OPTIMIZATION RESULTS


          Algorithm  Optimal Threshold   Recall  Precision  F1-Score
      Random Forest           0.718436 1.000000   0.446309  0.617169
            XGBoost           0.994738 1.000000   0.446309  0.617169
Logistic Regression           0.974135 0.984962   0.426016  0.594779
           LightGBM           0.000000 1.000000   0.003016  0.006014

IMPROVEMENT OVER BASELINE

Random For

In [18]:
# Cell 6: Feature Importance Analysis 
print("="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Get feature importance from tree-based models
print("\nExtracting feature importance from tree-based models...")

# LightGBM feature importance
lgbm_importance = pd.DataFrame({
    'Feature': feature_names,
    'LightGBM': lgbm_model.feature_importances_
})

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'Feature': feature_names,
    'XGBoost': xgb_model.feature_importances_
})

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': feature_names,
    'RandomForest': rf_model.feature_importances_
})

# Merge all importances
importance_df = lgbm_importance.merge(xgb_importance, on='Feature').merge(rf_importance, on='Feature')

# Calculate average importance
importance_df['Average'] = importance_df[['LightGBM', 'XGBoost', 'RandomForest']].mean(axis=1)

# Sort by average importance
importance_df = importance_df.sort_values('Average', ascending=False).reset_index(drop=True)

print("\nTop 15 Most Important Features:")
print(importance_df.head(15)[['Feature', 'Average', 'LightGBM', 'XGBoost', 'RandomForest']].to_string(index=False))

print("\n" + "="*80)
print("LOW-IMPORTANCE FEATURES (Bottom 10)")
print("="*80)
print("\n")
print(importance_df.tail(10)[['Feature', 'Average']].to_string(index=False))

# Identify features with very low importance (< 0.5% average)
low_importance_threshold = 0.005
low_importance_features = importance_df[importance_df['Average'] < low_importance_threshold]['Feature'].tolist()

print(f"\nFeatures with importance < {low_importance_threshold:.1%}: {len(low_importance_features)}")
if len(low_importance_features) > 0:
    print("Candidates for removal:")
    for feat in low_importance_features:
        avg_imp = importance_df[importance_df['Feature'] == feat]['Average'].values[0]
        print(f"  - {feat:40s} ({avg_imp:.4f})")

# Save importance analysis
importance_df.to_csv(OUTPUT_DIR + 'feature_importance_all_models.csv', index=False)
print(f"\nFeature importance saved: {OUTPUT_DIR}feature_importance_all_models.csv")


FEATURE IMPORTANCE ANALYSIS

Extracting feature importance from tree-based models...

Top 15 Most Important Features:
               Feature    Average  LightGBM  XGBoost  RandomForest
       filename_length 236.008281       708 0.000351      0.024493
            path_depth  94.687425       284 0.019051      0.043223
         is_executable  16.335642        49 0.000578      0.006349
      in_program_files  14.667499        44 0.000002      0.002494
     in_temp_directory  13.684095        41 0.000214      0.052070
   in_system_directory  10.671569        32 0.001196      0.013512
              is_image   9.676613        29 0.028871      0.000968
      timestamp_source   9.030744        27 0.000015      0.092216
     mft_time_modified   7.348893        22 0.002358      0.044319
modified_time_modified   7.020322        21 0.000158      0.060807
zero_in_nanoseconds_lf   7.001168        21 0.001459      0.002045
creation_time_modified   6.002850        18 0.003384      0.005165
  same_as_a

In [19]:
# Cell 7 : Retrain with Top Features Only 
print("="*80)
print("FEATURE SELECTION - RETRAIN WITH TOP FEATURES")
print("="*80)

# Test different feature set sizes
feature_counts = [30, 25, 20, 15, 10]

feature_selection_results = []

for n_features in feature_counts:
    print(f"\n{'='*80}")
    print(f"Testing with Top {n_features} Features")
    print(f"{'='*80}")
    
    # Select top N features
    top_features = importance_df.head(n_features)['Feature'].tolist()
    X_selected = X[top_features]
    
    print(f"\nSelected features: {', '.join(top_features[:5])}{'...' if n_features > 5 else ''}")
    
    # Retrain LightGBM (best baseline model)
    lgbm_selected = LGBMClassifier(
        n_estimators=100,
        scale_pos_weight=class_weight_ratio,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )
    lgbm_selected.fit(X_selected, y)
    proba_selected = lgbm_selected.predict_proba(X_selected)[:, 1]
    
    # Find optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y, proba_selected)
    
    f1_scores = []
    valid_thresholds = []
    
    for i, threshold in enumerate(thresholds):
        if recalls[i] >= TARGET_RECALL:
            f1 = 2 * (precisions[i] * recalls[i]) / (precisions[i] + recalls[i]) if (precisions[i] + recalls[i]) > 0 else 0
            f1_scores.append(f1)
            valid_thresholds.append(threshold)
    
    if len(f1_scores) > 0:
        best_idx = np.argmax(f1_scores)
        best_threshold = valid_thresholds[best_idx]
        
        y_pred_selected = (proba_selected >= best_threshold).astype(int)
        selected_recall = recall_score(y, y_pred_selected)
        selected_precision = precision_score(y, y_pred_selected)
        selected_f1 = f1_score(y, y_pred_selected)
        
        feature_selection_results.append({
            'Num Features': n_features,
            'Threshold': best_threshold,
            'Recall': selected_recall,
            'Precision': selected_precision,
            'F1-Score': selected_f1
        })
        
        print(f"  Optimal Threshold: {best_threshold:.4f}")
        print(f"  Recall:    {selected_recall:.4f}")
        print(f"  Precision: {selected_precision:.4f}")
        print(f"  F1-Score:  {selected_f1:.4f}")

selection_df = pd.DataFrame(feature_selection_results).sort_values('F1-Score', ascending=False)

print("\n" + "="*80)
print("FEATURE SELECTION RESULTS")
print("="*80)
print("\n")
print(selection_df.to_string(index=False))

# Find best feature count
best_selection_idx = selection_df['F1-Score'].idxmax()
best_n_features = selection_df.loc[best_selection_idx, 'Num Features']
best_selection_f1 = selection_df.loc[best_selection_idx, 'F1-Score']

print(f"\nBest Feature Count: {int(best_n_features)} (F1: {best_selection_f1:.4f})")


FEATURE SELECTION - RETRAIN WITH TOP FEATURES

Testing with Top 30 Features

Selected features: filename_length, path_depth, is_executable, in_program_files, in_temp_directory...
  Optimal Threshold: 0.0000
  Recall:    1.0000
  Precision: 0.0030
  F1-Score:  0.0060

Testing with Top 25 Features

Selected features: filename_length, path_depth, is_executable, in_program_files, in_temp_directory...
  Optimal Threshold: 0.0000
  Recall:    1.0000
  Precision: 0.0030
  F1-Score:  0.0060

Testing with Top 20 Features

Selected features: filename_length, path_depth, is_executable, in_program_files, in_temp_directory...
  Optimal Threshold: 0.0000
  Recall:    1.0000
  Precision: 0.0030
  F1-Score:  0.0060

Testing with Top 15 Features

Selected features: filename_length, path_depth, is_executable, in_program_files, in_temp_directory...
  Optimal Threshold: 1.0000
  Recall:    1.0000
  Precision: 0.0033
  F1-Score:  0.0066

Testing with Top 10 Features

Selected features: filename_length, pat

In [20]:
# Cell 8 : Summary of Training Optimization 
print("="*80)
print("TRAINING OPTIMIZATION SUMMARY")
print("="*80)

# Compile all results
summary_data = {
    'Approach': [
        'Baseline (0.5 threshold)',
        'Threshold Optimization',
        'Feature Selection'
    ],
    'Best Algorithm': [
        best_baseline_model,
        threshold_df.iloc[0]['Algorithm'],
        'LightGBM'
    ],
    'Recall': [
        baseline_df.iloc[0]['Recall'],
        threshold_df.iloc[0]['Recall'],
        selection_df.iloc[0]['Recall']
    ],
    'Precision': [
        baseline_df.iloc[0]['Precision'],
        threshold_df.iloc[0]['Precision'],
        selection_df.iloc[0]['Precision']
    ],
    'F1-Score': [
        baseline_df.iloc[0]['F1-Score'],
        threshold_df.iloc[0]['F1-Score'],
        selection_df.iloc[0]['F1-Score']
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n")
print(summary_df.to_string(index=False))

# Calculate improvements
baseline_f1 = summary_df.iloc[0]['F1-Score']
threshold_f1 = summary_df.iloc[1]['F1-Score']
selection_f1 = summary_df.iloc[2]['F1-Score']

print("\n" + "="*80)
print("IMPROVEMENTS")
print("="*80)

print(f"\nBaseline → Threshold Optimization:")
print(f"  F1: {baseline_f1:.4f} → {threshold_f1:.4f} ({(threshold_f1 - baseline_f1)/baseline_f1*100:+.2f}%)")
print(f"  Precision: {summary_df.iloc[0]['Precision']:.4f} → {summary_df.iloc[1]['Precision']:.4f} ({(summary_df.iloc[1]['Precision'] - summary_df.iloc[0]['Precision'])/summary_df.iloc[0]['Precision']*100:+.2f}%)")

print(f"\nBaseline → Feature Selection:")
print(f"  F1: {baseline_f1:.4f} → {selection_f1:.4f} ({(selection_f1 - baseline_f1)/baseline_f1*100:+.2f}%)")
print(f"  Precision: {summary_df.iloc[0]['Precision']:.4f} → {summary_df.iloc[2]['Precision']:.4f} ({(summary_df.iloc[2]['Precision'] - summary_df.iloc[0]['Precision'])/summary_df.iloc[0]['Precision']*100:+.2f}%)")

# Check if target met
print("\n" + "="*80)
print("TARGET ACHIEVEMENT (TRAINING DATA)")
print("="*80)

best_approach_idx = summary_df['F1-Score'].idxmax()
best_approach = summary_df.loc[best_approach_idx, 'Approach']
best_recall = summary_df.loc[best_approach_idx, 'Recall']
best_precision = summary_df.loc[best_approach_idx, 'Precision']
best_f1 = summary_df.loc[best_approach_idx, 'F1-Score']

print(f"\nBest Approach: {best_approach}")
print(f"  Recall:    {best_recall:.4f} ({'✓ PASS' if best_recall >= TARGET_RECALL else '✗ FAIL'})")
print(f"  Precision: {best_precision:.4f} ({'✓ PASS' if best_precision >= TARGET_PRECISION else '✗ FAIL'})")
print(f"  F1-Score:  {best_f1:.4f}")

if best_precision >= TARGET_PRECISION and best_recall >= TARGET_RECALL:
    print("\n✓ TARGET ACHIEVED on training data!")
    print("  Next: Validate on Lone Wolf held-out test set")
else:
    print("\n✗ TARGET NOT MET on training data")
    print("  Gap to target:")
    if best_precision < TARGET_PRECISION:
        print(f"    Precision: {best_precision:.4f} vs {TARGET_PRECISION:.4f} (need {TARGET_PRECISION - best_precision:+.4f})")
    if best_recall < TARGET_RECALL:
        print(f"    Recall: {best_recall:.4f} vs {TARGET_RECALL:.4f} (need {TARGET_RECALL - best_recall:+.4f})")


TRAINING OPTIMIZATION SUMMARY


                Approach Best Algorithm  Recall  Precision  F1-Score
Baseline (0.5 threshold)  Random Forest     1.0   0.446309  0.617169
  Threshold Optimization  Random Forest     1.0   0.446309  0.617169
       Feature Selection       LightGBM     1.0   0.003292  0.006562

IMPROVEMENTS

Baseline → Threshold Optimization:
  F1: 0.6172 → 0.6172 (+0.00%)
  Precision: 0.4463 → 0.4463 (+0.00%)

Baseline → Feature Selection:
  F1: 0.6172 → 0.0066 (-98.94%)
  Precision: 0.4463 → 0.0033 (-99.26%)

TARGET ACHIEVEMENT (TRAINING DATA)

Best Approach: Baseline (0.5 threshold)
  Recall:    1.0000 (✓ PASS)
  Precision: 0.4463 (✗ FAIL)
  F1-Score:  0.6172

✗ TARGET NOT MET on training data
  Gap to target:
    Precision: 0.4463 vs 0.7000 (need +0.2537)


In [21]:
# Cell 9: Prepare for Lone Wolf Validation 
print("="*80)
print("PREPARING FOR LONE WOLF VALIDATION")
print("="*80)

# Select best model configuration based on training results
best_config_idx = summary_df['F1-Score'].idxmax()
best_config = summary_df.loc[best_config_idx, 'Approach']

print(f"\nBest configuration: {best_config}")

# Determine which model and features to use for Lone Wolf
if best_config == 'Feature Selection':
    print(f"Using LightGBM with top {int(best_n_features)} features")
    best_features = importance_df.head(int(best_n_features))['Feature'].tolist()
    best_threshold = selection_df.loc[best_selection_idx, 'Threshold']
    
    # Train final model on selected features
    final_model = LGBMClassifier(
        n_estimators=100,
        scale_pos_weight=class_weight_ratio,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )
    final_model.fit(X[best_features], y)
    
elif best_config == 'Threshold Optimization':
    best_algo = threshold_df.iloc[0]['Algorithm']
    best_threshold = threshold_df.iloc[0]['Optimal Threshold']
    best_features = feature_names  # Use all features
    
    print(f"Using {best_algo} with threshold {best_threshold:.4f}")
    
    # Get the corresponding model
    if best_algo == 'LightGBM':
        final_model = lgbm_model
    elif best_algo == 'XGBoost':
        final_model = xgb_model
    elif best_algo == 'Random Forest':
        final_model = rf_model
    else:
        final_model = lr_model
        
else:  # Baseline
    best_algo = baseline_df.iloc[0]['Algorithm']
    best_threshold = 0.5
    best_features = feature_names
    
    print(f"Using {best_algo} with default threshold 0.5")
    
    if best_algo == 'LightGBM':
        final_model = lgbm_model
    elif best_algo == 'XGBoost':
        final_model = xgb_model
    elif best_algo == 'Random Forest':
        final_model = rf_model
    else:
        final_model = lr_model

print(f"\nFinal model:")
print(f"  Algorithm: {type(final_model).__name__}")
print(f"  Features: {len(best_features)}")
print(f"  Threshold: {best_threshold:.4f}")

print(f"\nFeatures used:")
for i, feat in enumerate(best_features[:10], 1):
    print(f"  {i:2d}. {feat}")
if len(best_features) > 10:
    print(f"  ... ({len(best_features)} total features)")

# Save configuration for Lone Wolf validation
config = {
    'model_type': type(final_model).__name__,
    'features': best_features,
    'threshold': float(best_threshold),
    'training_performance': {
        'recall': float(summary_df.loc[best_config_idx, 'Recall']),
        'precision': float(summary_df.loc[best_config_idx, 'Precision']),
        'f1_score': float(summary_df.loc[best_config_idx, 'F1-Score'])
    }
}

import json
config_path = OUTPUT_DIR + 'best_model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\nConfiguration saved: {config_path}")

# Save the final model
import pickle
model_path = OUTPUT_DIR + 'best_optimized_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print(f"Model saved: {model_path}")


PREPARING FOR LONE WOLF VALIDATION

Best configuration: Baseline (0.5 threshold)
Using Random Forest with default threshold 0.5

Final model:
  Algorithm: RandomForestClassifier
  Features: 30
  Threshold: 0.5000

Features used:
   1. zero_in_nanoseconds_lf
   2. zero_in_nanoseconds_suspicious
   3. zero_in_nanoseconds
   4. time_reversal_event
   5. basic_info_changed
   6. using_another_timestamp
   7. si_timestamp_changed
   8. update_resident_value
   9. creation_time_modified
  10. modified_time_modified
  ... (30 total features)

Configuration saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/best_model_config.json
Model saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/best_optimized_model.pkl


In [22]:
# Cell 10: Load and Prepare Lone Wolf Data
print("="*80)
print("LONE WOLF VALIDATION - LOADING DATA")
print("="*80)

# Load Lone Wolf data (same process as Prototype Tool Notebook 01)
print("\nLoading Lone Wolf artifacts...")

# Load LogFile
lf_df = pd.read_csv(LONEWOLF_LOGFILE)
print(f"  LogFile entries: {len(lf_df):,}")

# Load UsnJrnl
usn_df = pd.read_csv(LONEWOLF_USNJRNL)
print(f"  UsnJrnl entries: {len(usn_df):,}")

# Load ground truth
suspicious_df = pd.read_csv(LONEWOLF_SUSPICIOUS)
print(f"  Ground truth suspicious files: {len(suspicious_df):,}")

# Filter for suspicious events
print("\nFiltering for suspicious events...")

# LogFile: Time reversal events
if 'lf_detail' in lf_df.columns:
    lf_suspicious = lf_df[lf_df['lf_detail'].str.contains('Time Reversal', case=False, na=False)]
else:
    lf_suspicious = pd.DataFrame()

print(f"  LogFile suspicious events: {len(lf_suspicious):,}")

# UsnJrnl: Basic_Info_Changed events
if 'usn_event_info' in usn_df.columns:
    usn_suspicious = usn_df[usn_df['usn_event_info'].str.contains('Basic_Info_Changed', case=False, na=False)]
else:
    usn_suspicious = pd.DataFrame()

print(f"  UsnJrnl suspicious events: {len(usn_suspicious):,}")

print("\n✓ Lone Wolf data loaded")
print("\nNote: This is a simplified load. For full feature engineering, refer to Prototype Tool Notebooks 01-02")
print("      For this validation, we'll focus on the 12 known timestomped files from ground truth.")


LONE WOLF VALIDATION - LOADING DATA

Loading Lone Wolf artifacts...
  LogFile entries: 16,882
  UsnJrnl entries: 352,849
  Ground truth suspicious files: 15

Filtering for suspicious events...
  LogFile suspicious events: 0
  UsnJrnl suspicious events: 0

✓ Lone Wolf data loaded

Note: This is a simplified load. For full feature engineering, refer to Prototype Tool Notebooks 01-02
      For this validation, we'll focus on the 12 known timestomped files from ground truth.


In [23]:
# Cell 11: Extract Ground Truth Filenames 
print("="*80)
print("LONE WOLF GROUND TRUTH")
print("="*80)

# Extract filenames from suspicious CSV
# Format: detail column contains filenames like "DeathToll.jpg"
ground_truth_files = []

for idx, row in suspicious_df.iterrows():
    if row['source'] == 'usnjrnl' and row['category'] == 'Timestamp Manipulation':
        # Extract filename from detail
        detail = row['detail']
        # Pattern: "The timestamp of FILENAME may have been manipulated"
        if 'timestamp of' in detail:
            start = detail.find('timestamp of') + len('timestamp of ')
            end = detail.find('"', start)
            if end > start:
                filename = detail[start:end].strip()
                ground_truth_files.append(filename)

# Remove duplicates
ground_truth_files = list(set(ground_truth_files))
ground_truth_files.sort()

print(f"\nGround Truth: {len(ground_truth_files)} timestomped files")
print("\nKnown timestomped files:")
for i, filename in enumerate(ground_truth_files, 1):
    print(f"  {i:2d}. {filename}")

print("\nNote: For full Lone Wolf validation with complete feature engineering,")
print("      use the Prototype Tool pipeline (Notebooks 01-04)")
print("\nThis validation provides a simplified check of model performance")
print("on known timestomped files.")


LONE WOLF GROUND TRUTH

Ground Truth: 12 timestomped files

Known timestomped files:
   1. AIRPORT INFORMATION.docx
   2. BladeofGrass.jpg
   3. CubaDearmed.jpg
   4. DarkWolf.png
   5. DeathToll.jpg
   6. DemLogic.jpg
   7. HoldMyTidePod.jpg
   8. Huckleberry.png
   9. MyTiredHead.jpg
  10. Planning.docx
  11. RedGuns.jpg
  12. Sheep.jpg

Note: For full Lone Wolf validation with complete feature engineering,
      use the Prototype Tool pipeline (Notebooks 01-04)

This validation provides a simplified check of model performance
on known timestomped files.


In [24]:
# Cell 12: Decision Point - Next Steps 
print("="*80)
print("PHASE 4 OPTIMIZATION - DECISION POINT")
print("="*80)

print("\nTRAINING RESULTS:")
print(f"  Best Approach: {best_config}")
print(f"  Best F1-Score: {best_f1:.4f}")
print(f"  Best Precision: {best_precision:.4f}")
print(f"  Best Recall: {best_recall:.4f}")

print(f"\nTARGET:")
print(f"  Precision: ≥ {TARGET_PRECISION:.0%}")
print(f"  Recall: ≥ {TARGET_RECALL:.0%}")

# Assess if targets are met
precision_met = best_precision >= TARGET_PRECISION
recall_met = best_recall >= TARGET_RECALL

print("\n" + "="*80)
print("DECISION")
print("="*80)

if precision_met and recall_met:
    decision = "PROCEED_TO_VALIDATION"
    print("\n✓ TARGETS MET ON TRAINING DATA")
    print("\nNext Steps:")
    print("  1. Perform full Lone Wolf validation using Prototype Tool pipeline")
    print("  2. Apply optimized threshold and features")
    print("  3. Calculate final performance metrics")
    print("  4. Document results for thesis")
    
elif best_f1 > 0.65:
    decision = "PROCEED_WITH_CAUTION"
    print("\n⚠ TARGETS PARTIALLY MET")
    print(f"\nGaps:")
    if not precision_met:
        print(f"  Precision: {best_precision:.4f} vs {TARGET_PRECISION:.4f} (gap: {TARGET_PRECISION - best_precision:.4f})")
    if not recall_met:
        print(f"  Recall: {best_recall:.4f} vs {TARGET_RECALL:.4f} (gap: {TARGET_RECALL - best_recall:.4f})")
    
    print("\nOptions:")
    print("  A. Proceed to Lone Wolf validation (test set may perform better)")
    print("  B. Try extensive hyperparameter tuning of all 4 algorithms")
    print("  C. Accept current performance and document findings")
    
else:
    decision = "EXTENSIVE_TUNING_NEEDED"
    print("\n✗ TARGETS NOT MET")
    print(f"\nCurrent F1-Score ({best_f1:.4f}) is below acceptable threshold")
    
    print("\nRecommended Next Steps:")
    print("  1. Create Notebook 04: Extensive Hyperparameter Tuning (All Algorithms)")
    print("     - Deep grid search for Random Forest")
    print("     - Deep grid search for XGBoost")
    print("     - Deep grid search for LightGBM")
    print("     - Deep grid search for Logistic Regression")
    print("     - Expected time: 3-4 hours total")
    print("\n  2. Compare all tuned models")
    print("  3. Select best performer")
    print("  4. Validate on Lone Wolf")

# Save decision
decision_data = {
    'decision': decision,
    'best_config': best_config,
    'training_f1': float(best_f1),
    'training_precision': float(best_precision),
    'training_recall': float(best_recall),
    'targets_met': bool(precision_met and recall_met),  # Convert to Python bool
    'recommendation': 'Proceed to Lone Wolf validation' if (precision_met and recall_met) else 
                     'Consider extensive tuning or accept current performance'
}

decision_path = OUTPUT_DIR + 'optimization_decision.json'
with open(decision_path, 'w') as f:
    json.dump(decision_data, f, indent=2)

decision_path = OUTPUT_DIR + 'optimization_decision.json'
with open(decision_path, 'w') as f:
    json.dump(decision_data, f, indent=2)

print(f"\nDecision saved: {decision_path}")

print("\n" + "="*80)
print("PHASE 4 OPTIMIZATION COMPLETE")
print("="*80)


PHASE 4 OPTIMIZATION - DECISION POINT

TRAINING RESULTS:
  Best Approach: Baseline (0.5 threshold)
  Best F1-Score: 0.6172
  Best Precision: 0.4463
  Best Recall: 1.0000

TARGET:
  Precision: ≥ 70%
  Recall: ≥ 95%

DECISION

✗ TARGETS NOT MET

Current F1-Score (0.6172) is below acceptable threshold

Recommended Next Steps:
  1. Create Notebook 04: Extensive Hyperparameter Tuning (All Algorithms)
     - Deep grid search for Random Forest
     - Deep grid search for XGBoost
     - Deep grid search for LightGBM
     - Deep grid search for Logistic Regression
     - Expected time: 3-4 hours total

  2. Compare all tuned models
  3. Select best performer
  4. Validate on Lone Wolf

Decision saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/optimization_decision.json

PHASE 4 OPTIMIZATION COMPLETE


In [25]:
print("="*80)
print("LIGHTGBM PRECISION BOOST - TARGETING 95% RECALL")
print("="*80)

print("\nStrategy: Accept 95% recall to significantly improve precision")
print("This is acceptable for forensic triage where some false negatives")
print("can be caught through other means.\n")

# Retrain LightGBM with better parameters (from Notebook 01 results)
print("Retraining LightGBM with optimized parameters...")

lgbm_boosted = LGBMClassifier(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.1,
    scale_pos_weight=class_weight_ratio,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgbm_boosted.fit(X, y)
lgbm_boosted_proba = lgbm_boosted.predict_proba(X)[:, 1]

print("✓ LightGBM retrained with better parameters\n")

# Calculate precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y, lgbm_boosted_proba)

# Find thresholds that give different recall levels
target_recalls = [1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.90]

print("Precision vs Recall Trade-off:")
print("="*60)
print(f"{'Target Recall':<15} {'Threshold':<12} {'Actual Recall':<15} {'Precision':<12} {'F1-Score':<10}")
print("-"*60)

tradeoff_results = []

for target_recall in target_recalls:
    # Find threshold closest to target recall
    valid_indices = [i for i, r in enumerate(recalls) if r >= target_recall]
    
    if valid_indices:
        # Among valid thresholds, find the one with best precision
        best_idx = max(valid_indices, key=lambda i: precisions[i])
        threshold = thresholds[best_idx] if best_idx < len(thresholds) else 1.0
        
        # Calculate actual metrics
        y_pred = (lgbm_boosted_proba >= threshold).astype(int)
        actual_recall = recall_score(y, y_pred)
        actual_precision = precision_score(y, y_pred)
        actual_f1 = f1_score(y, y_pred)
        
        tradeoff_results.append({
            'Target Recall': target_recall,
            'Threshold': threshold,
            'Actual Recall': actual_recall,
            'Precision': actual_precision,
            'F1-Score': actual_f1
        })
        
        print(f"{target_recall:.0%}             {threshold:>8.4f}    {actual_recall:>8.4f}        {actual_precision:>8.4f}    {actual_f1:>8.4f}")

tradeoff_df = pd.DataFrame(tradeoff_results)

# Find best F1 at 95%+ recall
best_95_results = tradeoff_df[tradeoff_df['Actual Recall'] >= 0.95]

if len(best_95_results) > 0:
    best_95_idx = best_95_results['F1-Score'].idxmax()
    best_95_threshold = best_95_results.loc[best_95_idx, 'Threshold']
    best_95_recall = best_95_results.loc[best_95_idx, 'Actual Recall']
    best_95_precision = best_95_results.loc[best_95_idx, 'Precision']
    best_95_f1 = best_95_results.loc[best_95_idx, 'F1-Score']
    
    print("\n" + "="*80)
    print("OPTIMAL CONFIGURATION (≥95% RECALL)")
    print("="*80)
    
    print(f"\nBest threshold: {best_95_threshold:.4f}")
    print(f"  Recall:    {best_95_recall:.4f} ({int(best_95_recall * (y==1).sum())}/{(y==1).sum()})")
    print(f"  Precision: {best_95_precision:.4f}")
    print(f"  F1-Score:  {best_95_f1:.4f}")
    
    # Compare with baseline Random Forest
    rf_baseline_f1 = 0.6172
    improvement = best_95_f1 - rf_baseline_f1
    
    print(f"\nComparison with Baseline (Random Forest):")
    print(f"  Random Forest F1: {rf_baseline_f1:.4f}")
    print(f"  LightGBM F1:      {best_95_f1:.4f}")
    print(f"  Improvement:      {improvement:+.4f} ({improvement/rf_baseline_f1*100:+.2f}%)")
    
    # Check if target met
    print("\n" + "="*80)
    print("TARGET CHECK")
    print("="*80)
    
    recall_met = best_95_recall >= TARGET_RECALL
    precision_met = best_95_precision >= TARGET_PRECISION
    
    print(f"\nRecall:    {best_95_recall:.4f} {'✓ PASS' if recall_met else '✗ FAIL'} (target: {TARGET_RECALL:.0%})")
    print(f"Precision: {best_95_precision:.4f} {'✓ PASS' if precision_met else '✗ FAIL'} (target: {TARGET_PRECISION:.0%})")
    
    if precision_met and recall_met:
        print("\n✓✓✓ TARGET ACHIEVED! ✓✓✓")
        print("\nThis optimized LightGBM model can proceed to Lone Wolf validation!")
    else:
        precision_gap = max(0, TARGET_PRECISION - best_95_precision)
        print(f"\nPrecision gap to target: {precision_gap:.4f} ({precision_gap/TARGET_PRECISION*100:.1f}%)")
        
        if best_95_f1 > 0.65:
            print("Performance is close to target. Consider:")
            print("  1. Proceed to Lone Wolf validation (may perform better on test set)")
            print("  2. Try ensemble methods")
            print("  3. Accept current performance as reasonable for forensic triage")
    
    # Save the boosted model
    import pickle
    boosted_model_path = OUTPUT_DIR + 'lightgbm_precision_boosted.pkl'
    with open(boosted_model_path, 'wb') as f:
        pickle.dump(lgbm_boosted, f)
    
    boosted_config = {
        'model_type': 'LightGBM_Precision_Boosted',
        'threshold': float(best_95_threshold),
        'target_recall': 0.95,
        'performance': {
            'recall': float(best_95_recall),
            'precision': float(best_95_precision),
            'f1_score': float(best_95_f1)
        },
        'parameters': lgbm_boosted.get_params()
    }
    
    boosted_config_path = OUTPUT_DIR + 'lightgbm_boosted_config.json'
    with open(boosted_config_path, 'w') as f:
        json.dump(boosted_config, f, indent=2)
    
    print(f"\nBoosted model saved: {boosted_model_path}")
    print(f"Configuration saved: {boosted_config_path}")
    
else:
    print("\n⚠ WARNING: Could not find configuration with ≥95% recall")
    print("LightGBM calibration issue persists. Recommend extensive hyperparameter tuning.")


LIGHTGBM PRECISION BOOST - TARGETING 95% RECALL

Strategy: Accept 95% recall to significantly improve precision
This is acceptable for forensic triage where some false negatives
can be caught through other means.

Retraining LightGBM with optimized parameters...
✓ LightGBM retrained with better parameters

Precision vs Recall Trade-off:
Target Recall   Threshold    Actual Recall   Precision    F1-Score  
------------------------------------------------------------
100%               0.0000      1.0000          0.0030      0.0060
99%               0.0000      1.0000          0.0030      0.0060
98%               0.0000      1.0000          0.0030      0.0060
97%               0.0000      1.0000          0.0030      0.0060
96%               0.0000      1.0000          0.0030      0.0060
95%               0.0000      1.0000          0.0030      0.0060
94%               0.0000      1.0000          0.0030      0.0060
93%               0.0000      1.0000          0.0030      0.0060
92%       

In [28]:
print("="*80)
print("EXTENSIVE RANDOM FOREST HYPERPARAMETER TUNING (OPTIMIZED)")
print("="*80)

print("\nStrategy: Focused randomized search with reduced parameter space")
print("Goal: Achieve 80%+ F1 score in reasonable time (~20-30 minutes)")
print("Optimization: 30 iterations, max_estimators=300, 3-fold CV")
print("Trade-off: Still comprehensive but 10x faster\n")

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
import time

# Stage 1: Optimized Randomized Search
print("Stage 1: Randomized Search (Broad Exploration)")
print("-" * 80)

rf_param_distributions = {
    'n_estimators': [100, 150, 200, 250, 300],  # Reduced from 50-550
    'max_depth': [10, 20, 30, None],  # Reduced options
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.7],  # Reduced options
    'class_weight': ['balanced'],
    'bootstrap': [True],
    'max_samples': [0.8, 0.9]  # Reduced options
}

N_ITER = 30  # Reduced from 100
CV_FOLDS = 3  # Reduced from 5

print(f"\nParameter Space:")
print(f"  - n_estimators: {rf_param_distributions['n_estimators']}")
print(f"  - max_depth: {rf_param_distributions['max_depth']}")
print(f"  - min_samples_split: {rf_param_distributions['min_samples_split']}")
print(f"  - min_samples_leaf: {rf_param_distributions['min_samples_leaf']}")
print(f"  - max_features: {rf_param_distributions['max_features']}")
print(f"  - class_weight: {rf_param_distributions['class_weight']}")
print(f"  - max_samples: {rf_param_distributions['max_samples']}")
print(f"\nSearch Configuration:")
print(f"  - Iterations: {N_ITER}")
print(f"  - CV Folds: {CV_FOLDS}")
print(f"  - Total fits: {N_ITER * CV_FOLDS} (vs 500 in original)")
print(f"  - Estimated time: 20-30 minutes\n")

start_time = time.time()

rf_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_param_distributions,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring='f1',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2
)

print(f"Starting randomized search at {time.strftime('%H:%M:%S')}...")
print("This will take approximately 15-25 minutes...\n")

rf_random.fit(X, y)

random_duration = time.time() - start_time

print(f"\n{'='*80}")
print("RANDOMIZED SEARCH RESULTS")
print(f"{'='*80}")
print(f"Duration: {random_duration/60:.1f} minutes")
print(f"\nBest Parameters:")
for param, value in rf_random.best_params_.items():
    print(f"  {param}: {value}")

# Evaluate best model from randomized search
y_pred_random = rf_random.best_estimator_.predict(X)
random_recall = recall_score(y, y_pred_random)
random_precision = precision_score(y, y_pred_random)
random_f1 = f1_score(y, y_pred_random)

print(f"\nPerformance (Training Set):")
print(f"  Recall:    {random_recall:.4f}")
print(f"  Precision: {random_precision:.4f}")
print(f"  F1 Score:  {random_f1:.4f}")

# Stage 2: Focused Grid Search (only if randomized search improved)
print(f"\n{'='*80}")
print("Stage 2: Grid Search Refinement")
print(f"{'='*80}")

if random_f1 > 0.65:  # Only refine if we got decent results
    print("\nRandomized search found promising results. Refining...")
    
    # Build focused grid around best parameters
    best_params = rf_random.best_params_
    
    # Create narrow ranges around best values
    grid_params = {}
    
    # n_estimators: ±50 around best
    n_est_best = best_params['n_estimators']
    grid_params['n_estimators'] = sorted(list(set([
        max(50, n_est_best - 50),
        n_est_best,
        min(300, n_est_best + 50)
    ])))
    
    # max_depth: current + neighbors
    if best_params['max_depth'] is None:
        grid_params['max_depth'] = [30, None]
    else:
        depth_best = best_params['max_depth']
        grid_params['max_depth'] = sorted(list(set([
            max(5, depth_best - 10),
            depth_best,
            depth_best + 10,
            None
        ])))
    
    # min_samples_split: current value only
    grid_params['min_samples_split'] = [best_params['min_samples_split']]
    
    # min_samples_leaf: current + neighbors
    leaf_best = best_params['min_samples_leaf']
    grid_params['min_samples_leaf'] = sorted(list(set([
        max(1, leaf_best - 1),
        leaf_best,
        leaf_best + 1
    ])))
    
    # Keep best values for other params
    grid_params['max_features'] = [best_params['max_features']]
    grid_params['class_weight'] = [best_params['class_weight']]
    grid_params['bootstrap'] = [best_params['bootstrap']]
    grid_params['max_samples'] = [best_params['max_samples']]
    
    print(f"\nFocused Grid Search Parameters:")
    for param, values in grid_params.items():
        print(f"  {param}: {values}")
    
    total_combinations = 1
    for values in grid_params.values():
        total_combinations *= len(values)
    print(f"\nTotal combinations: {total_combinations}")
    print(f"Total fits: {total_combinations * CV_FOLDS}")
    print(f"Estimated time: {total_combinations * CV_FOLDS * 1.5 / 60:.1f} minutes\n")
    
    grid_start = time.time()
    
    rf_grid = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        param_grid=grid_params,
        cv=CV_FOLDS,
        scoring='f1',
        n_jobs=-1,
        verbose=2
    )
    
    print(f"Starting grid search at {time.strftime('%H:%M:%S')}...\n")
    rf_grid.fit(X, y)
    
    grid_duration = time.time() - grid_start
    
    # Evaluate grid search results
    y_pred_grid = rf_grid.best_estimator_.predict(X)
    grid_recall = recall_score(y, y_pred_grid)
    grid_precision = precision_score(y, y_pred_grid)
    grid_f1 = f1_score(y, y_pred_grid)
    
    print(f"\n{'='*80}")
    print("GRID SEARCH RESULTS")
    print(f"{'='*80}")
    print(f"Duration: {grid_duration/60:.1f} minutes")
    print(f"\nBest Parameters:")
    for param, value in rf_grid.best_params_.items():
        print(f"  {param}: {value}")
    
    print(f"\nPerformance (Training Set):")
    print(f"  Recall:    {grid_recall:.4f}")
    print(f"  Precision: {grid_precision:.4f}")
    print(f"  F1 Score:  {grid_f1:.4f}")
    
    # Choose best between randomized and grid
    if grid_f1 > random_f1:
        print("\nGrid search improved performance!")
        best_rf_model = rf_grid.best_estimator_
        best_rf_params = rf_grid.best_params_
        best_rf_f1 = grid_f1
        best_rf_precision = grid_precision
        best_rf_recall = grid_recall
        tuning_method = "Grid Search"
    else:
        print("\nRandomized search was better. Using those results.")
        best_rf_model = rf_random.best_estimator_
        best_rf_params = rf_random.best_params_
        best_rf_f1 = random_f1
        best_rf_precision = random_precision
        best_rf_recall = random_recall
        tuning_method = "Randomized Search"
else:
    print("\nRandomized search results below threshold. Skipping grid search.")
    print("Using randomized search results as final.")
    best_rf_model = rf_random.best_estimator_
    best_rf_params = rf_random.best_params_
    best_rf_f1 = random_f1
    best_rf_precision = random_precision
    best_rf_recall = random_recall
    tuning_method = "Randomized Search Only"

# Final Summary
total_duration = time.time() - start_time

print(f"\n{'='*80}")
print("EXTENSIVE RANDOM FOREST TUNING - FINAL RESULTS")
print(f"{'='*80}")

print(f"\nTotal Duration: {total_duration/60:.1f} minutes")
print(f"Best Method: {tuning_method}")

print(f"\nOptimal Hyperparameters:")
for param, value in sorted(best_rf_params.items()):
    print(f"  {param}: {value}")

print(f"\nFinal Performance (Training Set):")
print(f"  Recall:    {best_rf_recall:.4f} ({best_rf_recall*100:.2f}%)")
print(f"  Precision: {best_rf_precision:.4f} ({best_rf_precision*100:.2f}%)")
print(f"  F1 Score:  {best_rf_f1:.4f}")

# Compare to baseline
baseline_f1 = 0.617  # From Cell 7
improvement = best_rf_f1 - baseline_f1

print(f"\nImprovement over baseline:")
print(f"  Baseline F1:  {baseline_f1:.4f}")
print(f"  Tuned F1:     {best_rf_f1:.4f}")
print(f"  Improvement:  {improvement:+.4f} ({improvement/baseline_f1*100:+.1f}%)")

if best_rf_f1 >= 0.80:
    print("\n✓ SUCCESS: Achieved target F1 ≥ 0.80 for training set!")
    print("  This provides buffer for expected 10-15% drop on test set.")
elif best_rf_f1 >= 0.75:
    print("\n⚠ CLOSE: F1 ≥ 0.75 achieved. May need further tuning.")
else:
    print("\n✗ BELOW TARGET: F1 < 0.75. Consider:")
    print("  - Feature engineering (create interaction features)")
    print("  - Ensemble methods (stacking, voting)")
    print("  - Different algorithms (CatBoost, HistGradientBoosting)")

# Save results
rf_tuning_results = {
    'method': tuning_method,
    'best_params': {k: (int(v) if isinstance(v, (np.integer, np.int64)) else 
                       float(v) if isinstance(v, (np.floating, np.float64)) else 
                       str(v) if v is None else v) 
                   for k, v in best_rf_params.items()},
    'performance': {
        'recall': float(best_rf_recall),
        'precision': float(best_rf_precision),
        'f1_score': float(best_rf_f1)
    },
    'baseline_comparison': {
        'baseline_f1': float(baseline_f1),
        'improvement': float(improvement),
        'improvement_pct': float(improvement / baseline_f1 * 100)
    },
    'runtime': {
        'randomized_search_minutes': float(random_duration / 60),
        'total_minutes': float(total_duration / 60)
    }
}

output_path = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 4 - Hyperparameter Tuning/rf_extensive_tuning.json'
with open(output_path, 'w') as f:
    json.dump(rf_tuning_results, f, indent=2)

print(f"\nResults saved to: {output_path}")

# Store best model for later use
best_random_forest = best_rf_model
print("\nBest model stored in variable: best_random_forest")


EXTENSIVE RANDOM FOREST HYPERPARAMETER TUNING (OPTIMIZED)

Strategy: Focused randomized search with reduced parameter space
Goal: Achieve 80%+ F1 score in reasonable time (~20-30 minutes)
Optimization: 30 iterations, max_estimators=300, 3-fold CV
Trade-off: Still comprehensive but 10x faster

Stage 1: Randomized Search (Broad Exploration)
--------------------------------------------------------------------------------

Parameter Space:
  - n_estimators: [100, 150, 200, 250, 300]
  - max_depth: [10, 20, 30, None]
  - min_samples_split: [2, 5, 10]
  - min_samples_leaf: [1, 2, 4]
  - max_features: ['sqrt', 'log2', 0.7]
  - class_weight: ['balanced']
  - max_samples: [0.8, 0.9]

Search Configuration:
  - Iterations: 30
  - CV Folds: 3
  - Total fits: 90 (vs 500 in original)
  - Estimated time: 20-30 minutes

Starting randomized search at 13:49:56...
This will take approximately 15-25 minutes...

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END bootstrap=True, class_wei

## Save Best Model for Validation

Saving the tuned Random Forest model for use in Notebook 04.

In [29]:
import joblib
from pathlib import Path

# Create models directory if it doesn't exist
model_dir = Path('/Users/soni/Github/Digital-Detectives_Thesis/models')
model_dir.mkdir(parents=True, exist_ok=True)

# Save model
model_path = model_dir / 'phase4_best_random_forest.joblib'
joblib.dump(best_random_forest, model_path)

print(f"✓ Model saved: {model_path}")
print(f"  Model type: {type(best_random_forest).__name__}")
print(f"  Features expected: {best_random_forest.n_features_in_}")


✓ Model saved: /Users/soni/Github/Digital-Detectives_Thesis/models/phase4_best_random_forest.joblib
  Model type: RandomForestClassifier
  Features expected: 30
